In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS shopflow.silver")

In [0]:
spark.sql("SHOW SCHEMAS IN shopflow").show()

In [0]:
SILVER_PATH = "shopflow.silver"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DataType, IntegerType, DoubleType

In [0]:
df_bronze_customers = spark.table("shopflow.bronze.customers")

In [0]:
df_silver_customers = (
    df_bronze_customers
    .withColumn("name",    F.trim(F.col("name")))
    .withColumn("email",   F.lower(F.trim(F.col("email"))))
    .withColumn("city",    F.trim(F.col("city")))
    .withColumn("country", F.trim(F.col('country')))

    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-mm-dd"))

    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id","email"])

    .withColumn("prcessed_at", F.current_timestamp())
)

In [0]:
df_silver_customers.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("shopflow.silver.customers")

df_silver_customers.show(3)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DataType, IntegerType, DoubleType
df_bronze_products = spark.table("shopflow.bronze.products")
df_silver_prodcts = (
    df_bronze_products
    
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("base_price", F.col("base_price").cast(DoubleType()))

    .withColumn("product_name", F.trim(F.col("product_name")))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))

    .filter(F.col("base_price")> 0)

    .dropDuplicates(["product_id"])
    .dropna(subset=["product_id","product_name"])

    .withColumn("processed_at", F.current_timestamp())


)

In [0]:
df_silver_prodcts.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("shopflow.silver.product")

df_bronze_products.show(3)

In [0]:
df_bronze_orders = spark.table("shopflow.bronze.orders")
VALID_STATUS = ["completed","pending", "returned"]

df_silver_orders = (
    df_bronze_orders
    .withColumn("order_id", F.col("order_id").cast(IntegerType()))
    .withColumn("customer_id", F.col("customer_id").cast(IntegerType()))
    .withColumn("product_id", F.col("product_id").cast(IntegerType()))
    .withColumn("quantity", F.col("quantity").cast(IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(DoubleType()))
    .withColumn("total_price", F.col("total_price").cast(DoubleType()))
    .withColumn("order_date", F.to_date(F.col("order_date"), "yyyy-mm-dd"))

    .withColumn("status", F.lower(F.trim(F.col("status"))))

    .filter(F.col("quantity")> 0)
    .filter(F.col("total_price") > 0)
    .filter(F.col("status").isin(VALID_STATUS))

    .dropna(subset =["order_id", "customer_id", "product_id", "order_date"])
    .dropDuplicates(["order_id"])

    .withColumn("revenue", F.round(F.col("quantity")* F.col("unit_price"),2))
    .withColumn("order_month",
                F.date_format(F.col("order_date"), "yyyy-MM"))
    .withColumn("order_year", F.year(F.col("order_date")))

    .withColumn("processed_at", F.current_timestamp())
)

In [0]:
df_silver_orders.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable("shopflow.silver.orders")
df_silver_orders.show(3)

In [0]:
print("Silver row count")

for tbl in["shopflow.silver.customers",
           "shopflow.silver.product",
           "shopflow.silver.orders"]:
    spark.sql(f"SELECT '{tbl}' AS tbl, COUNT(*) as rows FROM {tbl}").show()

In [0]:
print("Null check on oreder")
spark.sql("""
          SELECT
        SUM(CASE WHEN order_id   IS NULL THEN 1 ELSE 0 END) AS null_order_id,
        SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS null_customer,
        SUM(CASE WHEN order_date  IS NULL THEN 1 ELSE 0 END) AS null_date,
        SUM(CASE WHEN revnew     IS NULL THEN 1 ELSE 0 END) AS null_revenue
    FROM shopflow.silver.orders""")

In [0]:
print("\n=== Status distribution ===")
spark.sql("""
    SELECT status, COUNT(*) as orders,
           ROUND(SUM(revnew),2) as total_revenue
    FROM shopflow.silver.orders
    GROUP BY status ORDER BY total_revenue DESC
""").show()

In [0]:
print("\n=== Sample enriched orders ===")
spark.sql("""
    SELECT order_id, order_date, order_month,
           quantity, unit_price, revnew, status
    FROM shopflow.silver.orders LIMIT 5
""").show()